In [ ]:
from pathlib import Path
import re
from typing import List, Dict

In [ ]:
# ==========================
# Configuration
# ==========================

PROJECT_ROOT = Path.cwd()

PATIENT_CONTEXTS_DIR = PROJECT_ROOT / "patient_contexts"


In [ ]:
# ==========================
# Regex Patterns
# ==========================

# Pattern to capture section titles
section_header_pattern = r"=+\r?\n(.*?)\r?\n=+"
section_header_regex = re.compile(section_header_pattern)

radiology_report_pattern = r"\{'charttime': Timestamp\('([^']+)'\)\}"
radiology_report_regex = re.compile(radiology_report_pattern)


In [ ]:
FREE_TEXT_SECTIONS = {
    "HISTORY OF PRESENT ILLNESS",
    "PHYSICAL EXAMINATION",
    "RADIOLOGY REPORTS"
}

TABULAR_SECTIONS = {
    "PATIENT DEMOGRAPHICS",
    "ADMISSION",
    "SERVICES",
    "TRANSFERS",
    "PROCEDURES",
    "MEDICATIONS",
    "MEDICATION ADMINISTRATION",
    "LABORATORY RESULTS",
    "MICROBIOLOGY",
    "EMERGENCY DEPARTMENT DATA",
    "ICU SUMMARY",
    "PATIENT CHEMISTRY",
    "COMPLETE BLOOD COUNT",
    "BLOOD DIFFERENTIAL",
    "COAGULATION",
    "BLOOD GAS",
    "CARDIAC MARKERS",
    "ENZYMES",
    "PATIENT VITALS",
    "URINE OUTPUT",
    "VENTILATION",
    "OXYGEN DELIVERY",
    "INPUT EVENTS",
    "PROCEDURE EVENTS",
    "OUTPUT EVENTS",
    "PATIENT ECG"
}

In [ ]:
patient_files = sorted(PATIENT_CONTEXTS_DIR.glob("patient_*.txt"))

print(f"Found {len(patient_files)} files.")

for file in patient_files[:5]:
    print(file.name)

In [ ]:
# storing the patient case input files in the patient_case_files variable
patient_case_files = sorted(
    f for f in PATIENT_CONTEXTS_DIR.glob("patient_*.txt")
    if not f.name.endswith("_target.txt")
)
print(f"Found {len(patient_case_files)} patient case(s).")

for file in patient_case_files:
    print(file.name)

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download("punkt_tab")

In [ ]:
# !pip show sentence-transformers scikit-learn numpy
# !pip install -U scikit-learn
# !pip install -U sentence-transformers

%pip uninstall -y scikit-learn
%pip install --no-cache-dir --force-reinstall scikit-learn
%pip install --no-cache-dir --upgrade sentence-transformers

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
import numpy as np

def load_patient_case(filepath: Path) -> str:
    """Load a patient case text file and return its contents."""

    with open(filepath, "r", encoding="utf-8") as file:
        content = file.read()

    return content



def extract_ids(filepath: Path) -> tuple[int, int]:
    """
    Extract subject_id and hadm_id from a patient case filename.

    Parameters
    ----------
    filepath : Path
        Path to the patient case file.

    Returns
    -------
    tuple
        (subject_id, hadm_id)
    """

    match = re.search(r"patient_(\d+)_(\d+)\.txt$", filepath.name)

    if not match:
        raise ValueError(f"Invalid patient filename: {filepath.name}")

    subject_id = int(match.group(1))
    hadm_id = int(match.group(2))

    return subject_id, hadm_id




def find_section_headers(content: str):
    """
    Find all section headers in a patient case.

    Parameters
    ----------
    content : str
        The patient case text.

    Returns
    -------
    list
        A list of regex match objects.
    """
    return list(section_header_regex.finditer(content))




def extract_sections(content: str) -> list:
    """
    Extract all clinical sections from a patient case.

    Parameters
    ----------
    content : str
        Complete patient case text.

    Returns
    -------
    list
        A list of dictionaries, where each dictionary contains:
        - section : section title
        - content : section content
    """

    matches = find_section_headers(content)

    sections = []

    for i, match in enumerate(matches):

        # Extract the section title
        section_title = match.group(1).strip()

        # Determine where the section content starts
        content_start = match.end()

        # Determine where the section content ends
        if i < len(matches) - 1:
            content_end = matches[i + 1].start()
        else:
            content_end = len(content)

        # Extract the section content
        section_content = content[content_start:content_end].strip()

        # Skip empty sections
        if not section_content:
            continue

        # Store the extracted section
        sections.append({
            "section": section_title,
            "content": section_content
        })

    return sections



def split_radiology_reports(content: str) -> list:
    """
    Split the radiology section into individual report events.

    Parameters
    ----------
    content : str
        The content of the Radiology Reports section.

    Returns
    -------
    list
        A list where each element is one complete radiology report.
    """

    matches = list(radiology_report_regex.finditer(content))

    reports = []

    for i, match in enumerate(matches):

        #report_start = match.start()
        charttime = match.group(1).strip()
        report_start = match.end()

        if i < len(matches) - 1:
            report_end = matches[i + 1].start()
        else:
            report_end = len(content)

        report_text = content[report_start:report_end].strip()

        if report_text:
            #reports.append(report)
            reports.append({
                "charttime": charttime,
                "text": report_text })

    return reports





def load_embedding_model():
    """
    Load the sentence embedding model.
    """
    return SentenceTransformer("BAAI/bge-base-en-v1.5")




from transformers import AutoTokenizer

def load_tokenizer():
    """
    Load the tokenizer matching the embedding model, used to enforce
    the model's real token-length ceiling during chunking.
    """
    return AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")


def token_count(text: str) -> int:
    """
    Count tokens using the embedding model's own tokenizer, rather than
    a whitespace word count, since the model's context window is a
    token limit, not a word limit.
    """
    return len(tokenizer.encode(text, add_special_tokens=False))







def embed_sentences(sentences: list):
    """
    Generate embeddings for a list of sentences.
    """
    return embedding_model.encode(
        sentences,
        convert_to_numpy=True,
        normalize_embeddings=True
    )



# def semantic_merge(sentences: list, sentence_embeddings, section, subject_id, hadm_id, parent_id=None, similarity_threshold: float = 0.8,max_chunk_words: int = 500) -> list:
#     if not sentences:
#         return []
#     n = len(sentences)
#     chunks = []
#     current_chunk = [sentences[0]]
#     for i in range(1, n):
#         similarity = np.dot(sentence_embeddings[i - 1], sentence_embeddings[i])
#         candidate_chunk = current_chunk + [sentences[i]]
#         candidate_word_count = len(" ".join(candidate_chunk).split())

#         if (similarity >= similarity_threshold and candidate_word_count <= max_chunk_words):
#             current_chunk.append(sentences[i])

#         else:
#             chunks.append( add_metadata(chunk_text=" ".join(current_chunk), subject_id=subject_id, hadm_id=hadm_id, section=section["section"], chunk_type="adaptive", parent_id=parent_id if parent_id is not None else section["section"]) )
#             current_chunk = [sentences[i]]
#     if current_chunk:
#         chunks.append( add_metadata(chunk_text=" ".join(current_chunk), subject_id=subject_id, hadm_id=hadm_id, section=section["section"], chunk_type="adaptive", parent_id=parent_id if parent_id is not None else section["section"]) )

#     return chunks





import re

# Matches a sentence that is ONLY a bare list marker, e.g. "1.", "2)", "a.", "(3)"
BARE_MARKER_PATTERN = re.compile(r'^\(?[0-9]+[\.\)]$|^\(?[a-zA-Z][\.\)]$')

# Matches a short "label: N." stub, e.g. "Past Medical History: 1."
LABELED_STUB_PATTERN = re.compile(r'^[\w\s]+:\s*\d+\.$')
LABELED_STUB_MAX_WORDS = 6


def is_orphan_list_fragment(sentence: str) -> bool:
    """
    True if this sentence is a numbering artifact that shouldn't stand
    alone: either a bare marker ("1.") or a short "label: N." stub
    ("Past Medical History: 1.").
    """
    text = sentence.strip()

    if BARE_MARKER_PATTERN.match(text):
        return True

    if LABELED_STUB_PATTERN.match(text) and len(text.split()) <= LABELED_STUB_MAX_WORDS:
        return True

    return False


def repair_orphan_list_fragments(sentences: list) -> list:
    """
    Post-processes a sentence list from sent_tokenize to fuse numbering
    artifacts into the following sentence, since NLTK sometimes splits
    on 'N.' as if it were sentence-ending punctuation.

    Handles:
      - "Past Medical History: 1." + "CARDIAC RISK FACTORS..."
        -> "Past Medical History: 1. CARDIAC RISK FACTORS..."
      - "The patient has the following symptoms." + "1." + "high temperature"
        -> "The patient has the following symptoms." + "1. high temperature"
    """

    if not sentences:
        return sentences

    repaired = []
    i = 0
    n = len(sentences)

    while i < n:
        current = sentences[i].strip()

        if is_orphan_list_fragment(current) and i + 1 < n:
            # Fuse forward into the next sentence, since a bare/labeled
            # marker is meaningless without what follows it.
            merged = current + " " + sentences[i + 1].strip()
            repaired.append(merged)
            i += 2  # consumed both this fragment and the next sentence
            continue

        if is_orphan_list_fragment(current) and i + 1 >= n:
            # Last sentence in the section and still an orphan marker:
            # nothing to merge forward into, so fuse it backward instead
            # rather than leaving it as a standalone meaningless chunk.
            if repaired:
                repaired[-1] = repaired[-1] + " " + current
            else:
                repaired.append(current)
            i += 1
            continue

        repaired.append(current)
        i += 1

    return repaired




def semantic_merge(sentences: list, sentence_embeddings, section, subject_id, hadm_id, parent_id=None, similarity_threshold: float = 0.57, max_chunk_tokens: int = 480) -> list:
    """
    Merge sentences into semantically coherent chunks using the
    aggregate embedding of the current chunk.
    (the target is to allow 500 tokens in a chunk as maximum ceilling, based on the model embedding window size, but a margin has been left for allowing the addition of the model's special tokens).
    """

    if not sentences:
        return []

    # Precompute each sentence's token count once, instead of re-tokenizing
    # the whole accumulated chunk on every comparison.
    sentence_token_counts = [token_count(s) for s in sentences]
    
    chunks = []

    current_chunk = [sentences[0]]
    current_chunk_token_count = sentence_token_counts[0]

    # Index of the first sentence in the current chunk
    current_chunk_start = 0

    n = len(sentences)

    for i in range(1, n):

        # Compute the aggregate embedding of the current chunk
        current_chunk_embedding = np.mean(
            sentence_embeddings[current_chunk_start:i],
            axis=0
        )

        # Re-normalize after averaging
        current_chunk_embedding /= np.linalg.norm(current_chunk_embedding)

        # Cosine similarity
        similarity = np.dot(
            current_chunk_embedding,
            sentence_embeddings[i]
        )
        candidate_token_count = current_chunk_token_count + sentence_token_counts[i]
        # candidate_chunk = current_chunk + [sentences[i]]
        # candidate_word_count = len(" ".join(candidate_chunk).split())

        if (
            similarity >= similarity_threshold
            and candidate_token_count <= max_chunk_tokens   #renamed from max_chunk_words
        ):
            current_chunk.append(sentences[i])
            current_chunk_token_count = candidate_token_count

        else:

            chunks.append(
                add_metadata(
                    chunk_text=" ".join(current_chunk),
                    subject_id=subject_id,
                    hadm_id=hadm_id,
                    section=section["section"],
                    chunk_type="adaptive",
                    parent_id=parent_id if parent_id is not None else section["section"]
                )
            )

            # Start a new chunk
            current_chunk = [sentences[i]]
            current_chunk_token_count = sentence_token_counts[i]
            current_chunk_start = i

    # Save the final chunk
    if current_chunk:
        chunks.append(
            add_metadata(
                chunk_text=" ".join(current_chunk),
                subject_id=subject_id,
                hadm_id=hadm_id,
                section=section["section"],
                chunk_type="adaptive",
                parent_id=parent_id if parent_id is not None else section["section"]
            )
        )

    return chunks



    
    
def adaptive_radiology_chunk(section, subject_id, hadm_id): 
    chunks = []
    reports= split_radiology_reports(section["content"])
    for report in reports:
        charttime = report["charttime"]
        report_text = report["text"]
        # The next two lines are to process the newline which is already present in the original report text. It is quite common in MIMIC-IV notes because they are often wrapped at around 80 characters,
        # where the newlines are formatting rather than semantic boundaries. For this we reformt the new lines and replace them with spaces.
        # However, Replacing every \n with a space would also remove intentional paragraph breaks, not just line wrapping. So I'll hold with this step 
        # report_text = re.sub(r"\n+", " ", report_text)
        # report_text = re.sub(r"\s+", " ", report_text).strip()
        sentences = sent_tokenize(report_text)
        
        # Remove punctuation-only sentences
        sentences = [s for s in sentences if re.search(r"[A-Za-z0-9]", s)]

        #To repair orphan number in a chunk or orphan list fragment
        sentences = repair_orphan_list_fragments(sentences)
    
        sentence_embeddings = embed_sentences(sentences)
        report_chunks = semantic_merge(sentences, sentence_embeddings, section, subject_id, hadm_id, parent_id=charttime)
        chunks.extend(report_chunks)
    return chunks






def adaptive_text_chunk(section, subject_id, hadm_id): 

    text = section["content"]
    # The next two lines are to process the newline which is already present in the original report text. It is quite common in MIMIC-IV notes because they are often wrapped at around 80 characters,
    # where the newlines are formatting rather than semantic boundaries. For this we reformt the new lines and replace them with spaces.
    # However, Replacing every \n with a space would also remove intentional paragraph breaks, not just line wrapping. So I'll hold with this step. 
    # text = re.sub(r"\n+", " ", text)
    # text = re.sub(r"\s+", " ", text).strip()

    sentences = sent_tokenize(text)

    # Remove punctuation-only sentences
    sentences = [s for s in sentences if re.search(r"[A-Za-z0-9]", s)]

    #To repair orphan number in a chunk or orphan list fragment
    sentences = repair_orphan_list_fragments(sentences)
    
    sentence_embeddings = embed_sentences(sentences)

    chunks = semantic_merge(sentences, sentence_embeddings, section, subject_id, hadm_id)
    return chunks




def adaptive_chunk(section, subject_id, hadm_id):

    if section["section"] == "RADIOLOGY REPORTS":
        return adaptive_radiology_chunk(section, subject_id, hadm_id)

    return adaptive_text_chunk(section, subject_id, hadm_id)


def table_chunk(section, subject_id, hadm_id, max_chunk_tokens: int = 480) -> list:
    """
    Chunk a tabular section while preserving the table header in every chunk.

    Parameters
    ----------
    section : dict
        Dictionary containing:
            - section : section title
            - content : table content

    max_chunk_tokens : int
        Maximum number of tokens allowed in a chunk 
        (the target is to allow 500 tokens in a chunk as maximum ceilling, based on the model embedding window size, but a margin has been left for allowing the addition of the model's special tokens).

    Returns
    -------
    list
        List of table chunks.
    """

    lines = section["content"].splitlines()

    if len(lines) < 2:
        return []

    header = lines[0].strip()
    rows = [row.strip() for row in lines[1:] if row.strip()]

    header_token_count = token_count(header)

    chunks = []

    current_rows = []
    current_token_count = header_token_count

    for row in rows:

        row_token_count = token_count(row)
        candidate_token_count = current_token_count + row_token_count

        if candidate_token_count <= max_chunk_tokens:
            current_rows.append(row)
            current_token_count = candidate_token_count

        else:

            if not current_rows:
                chunks.append(add_metadata(chunk_text="\n".join([header, row]), subject_id=subject_id, hadm_id=hadm_id, section=section["section"], chunk_type="table", parent_id=section["section"]))
                current_token_count = header_token_count
                continue

            chunks.append(add_metadata(chunk_text="\n".join([header] + current_rows), subject_id=subject_id, hadm_id=hadm_id, section=section["section"], chunk_type="table", parent_id=section["section"]))

            current_rows = [row]
            current_token_count = header_token_count + row_token_count

    if current_rows:
        chunks.append(add_metadata(chunk_text="\n".join([header] + current_rows), subject_id=subject_id, hadm_id=hadm_id, section=section["section"], chunk_type="table", parent_id=section["section"]))

    return chunks






def add_metadata( chunk_text: str, subject_id: int, hadm_id: int, section: str, chunk_type: str, parent_id: str = None) -> dict:
    """
    Attach metadata to a chunk.

    Parameters
    ----------
    chunk_text : str
        The chunk text.

    subject_id : int
        Patient identifier.

    hadm_id : int
        Hospital admission identifier.

    section : str
        Clinical section name.

    chunk_type : str
        Either "adaptive" or "table".

    parent_id : str, optional
        Parent document/event identifier.

    Returns
    -------
    dict
        Chunk with metadata.
    """

    return {
        "subject_id": subject_id,
        "hadm_id": hadm_id,
        "section": section,
        "chunk_type": chunk_type,
        "parent_id": parent_id,
        "text": chunk_text
    }



def assign_chunk_ids(chunks: list) -> None:
    """
    Assign sequential chunk IDs to a patient's chunks.

    Parameters
    ----------
    chunks : list
        List of chunk dictionaries.

    Returns
    -------
    None
    """

    for i, chunk in enumerate(chunks, start=1):
        chunk["chunk_id"] = i




    
    

In [ ]:
# embedding_model = load_embedding_model()

# sentences = [
#     "The patient presented with chest pain.",
#     "Troponin was elevated."
# ]

# embeddings = embed_sentences(sentences)

# print(type(embeddings))
# print(embeddings.shape)

In [ ]:
# import sklearn
# import numpy
# import scipy

# print("scikit-learn:", sklearn.__version__)
# print("numpy:", numpy.__version__)
# print("scipy:", scipy.__version__)
# print("sklearn path:", sklearn.__file__)

# from sklearn.utils._metadata_requests import _raise_for_params

# print("scikit-learn import OK")

In [ ]:
# from sentence_transformers import SentenceTransformer

# print("sentence-transformers import OK")

In [ ]:
def analyse_chunk_sizes(chunks: list):
    """
    Analyse the word-count distribution of chunks.
    """

    word_counts = [len(chunk["text"].split()) for chunk in chunks]
    print("********* Analyse the word-count distribution of chunks **********")
    print(f"Number of chunks : {len(word_counts)}")
    print(f"Minimum words    : {min(word_counts)}")
    print(f"Maximum words    : {max(word_counts)}")
    print(f"Average words    : {sum(word_counts) / len(word_counts):.1f}")

    word_counts = sorted(word_counts)

    median = word_counts[len(word_counts) // 2]
    print(f"Median words     : {median}")

    for threshold in [10, 20, 30, 40, 50]:
        count = sum(w <= threshold for w in word_counts)
        percentage = 100 * count / len(word_counts)

        print(
            f"Chunks ≤ {threshold:2d} words : "
            f"{count:3d} ({percentage:.1f}%)"
        )

In [ ]:
#### Main ####

embedding_model = load_embedding_model()
tokenizer = load_tokenizer()


all_patient_chunks = []   # accumulates across every patient

for file in patient_case_files:

    all_chunks = []
    
    subject_id, hadm_id = extract_ids(file)
    print (f"subject_id: {subject_id}")
    
    content = load_patient_case(file)

    sections = extract_sections(content)

    for section in sections:

        if section["section"] in FREE_TEXT_SECTIONS:
            chunks = adaptive_chunk(section, subject_id, hadm_id)
            print (f"section: {section['section']}")

        elif section["section"] in TABULAR_SECTIONS:
            chunks = table_chunk(section, subject_id, hadm_id)
            print (f"section: {section['section']}")

        else:
            raise ValueError(f"Unknown section: {section['section']}")

        all_chunks.extend(chunks)
        
    assign_chunk_ids(all_chunks)
    all_patient_chunks.extend(all_chunks)   # <-- accumulate here
    
    print(f"Total chunks: {len(all_chunks)}")

    # for i, chunk in enumerate(all_chunks, start=1):
    #     print("=" * 80)
    #     print(f"Chunk {i}")
    #     print("=" * 80)
    #     print(chunk)
    #     print()

# analyse_chunk_sizes(all_chunks)
analyse_chunk_sizes(all_patient_chunks)

# save once, after the loop
import json
with open("all_patient_chunks.jsonl", "w") as f:
    for c in all_patient_chunks:
        f.write(json.dumps(c) + "\n")
        

In [ ]:
from collections import defaultdict

section_chunks = defaultdict(list)

#for chunk in all_chunks:
for chunk in all_patient_chunks:
    section_chunks[chunk["section"]].append(chunk)

for section_name, chunks in section_chunks.items():

    print("=" * 80)
    print(section_name)
    print("=" * 80)

    analyse_chunk_sizes(chunks)
    print()

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

def collect_similarities_for_sentences(sentences, sentence_embeddings):
    """Replays semantic_merge's similarity computation without merging,
    just to record the raw similarity values it would decide on."""
    sims = []
    if len(sentences) < 2:
        return sims

    current_chunk_start = 0
    for i in range(1, len(sentences)):
        current_chunk_embedding = np.mean(
            sentence_embeddings[current_chunk_start:i], axis=0
        )
        current_chunk_embedding /= np.linalg.norm(current_chunk_embedding)
        similarity = np.dot(current_chunk_embedding, sentence_embeddings[i])
        sims.append(similarity)
        # mimic "always start new chunk at this sentence" so we sample
        # the similarity space broadly rather than only within long runs
        current_chunk_start = i

    return sims


section_similarities = defaultdict(list)

SAMPLE_SIZE = 30  # bump up if you want a fuller distribution; start small to check runtime
sample_files = patient_case_files[:SAMPLE_SIZE]

for file in sample_files:
    subject_id, hadm_id = extract_ids(file)
    content = load_patient_case(file)
    sections = extract_sections(content)

    for section in sections:
        if section["section"] not in FREE_TEXT_SECTIONS:
            continue

        if section["section"] == "RADIOLOGY REPORTS":
            reports = split_radiology_reports(section["content"])
            texts = [r["text"] for r in reports]
        else:
            texts = [section["content"]]

        for text in texts:
            sentences = sent_tokenize(text)
            sentences = [s for s in sentences if re.search(r"[A-Za-z0-9]", s)]
            if len(sentences) < 2:
                continue
            embeddings = embed_sentences(sentences)
            sims = collect_similarities_for_sentences(sentences, embeddings)
            section_similarities[section["section"]].extend(sims)


# ---- Plot + percentile table ----
fig, axes = plt.subplots(1, len(section_similarities), figsize=(6 * len(section_similarities), 4))
if len(section_similarities) == 1:
    axes = [axes]

for ax, (section_name, sims) in zip(axes, section_similarities.items()):
    ax.hist(sims, bins=40)
    ax.set_title(f"{section_name}\n(n={len(sims)})")
    ax.set_xlabel("cosine similarity")
    for thresh in [0.5, 0.55, 0.6, 0.65, 0.7]:
        ax.axvline(thresh, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

print(f"{'Section':<30}{'p25':>8}{'p50':>8}{'p75':>8}{'p90':>8}")
for section_name, sims in section_similarities.items():
    sims_sorted = sorted(sims)
    n = len(sims_sorted)
    p25 = sims_sorted[int(n * 0.25)]
    p50 = sims_sorted[int(n * 0.50)]
    p75 = sims_sorted[int(n * 0.75)]
    p90 = sims_sorted[int(n * 0.90)]
    print(f"{section_name:<30}{p25:8.3f}{p50:8.3f}{p75:8.3f}{p90:8.3f}")